In [1]:
import polars as pl

# Load dataset safely
df = pl.read_csv(
    "../data/data.csv",
    encoding="latin-1",
    ignore_errors=True
)

df.head()

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
i64,str,str,i64,str,f64,i64,str
536365,"""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,"""12/1/2010 8:26""",2.55,17850,"""United Kingdom"""
536365,"""71053""","""WHITE METAL LANTERN""",6,"""12/1/2010 8:26""",3.39,17850,"""United Kingdom"""
536365,"""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,"""12/1/2010 8:26""",2.75,17850,"""United Kingdom"""
536365,"""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,"""12/1/2010 8:26""",3.39,17850,"""United Kingdom"""
536365,"""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,"""12/1/2010 8:26""",3.39,17850,"""United Kingdom"""


In [2]:
df = df.with_columns(
    pl.col("InvoiceDate").str.to_datetime("%m/%d/%Y %H:%M")
)

In [3]:
df = df.drop_nulls("CustomerID")

In [4]:
df = df.with_columns(
    (pl.col("Quantity") * pl.col("UnitPrice")).alias("Revenue")
)

In [5]:
# Get latest date in dataset
snapshot_date = df.select(pl.col("InvoiceDate").max()).to_series()[0]

# Create RFM
rfm = df.group_by("CustomerID").agg([
    (snapshot_date - pl.col("InvoiceDate").max()).dt.total_days().alias("Recency"),
    pl.col("InvoiceNo").n_unique().alias("Frequency"),
    pl.col("Revenue").sum().alias("Monetary")
])

rfm.head()

CustomerID,Recency,Frequency,Monetary
i64,i64,u32,f64
13767,1,38,16945.71
14824,9,4,1127.71
12353,203,1,89.0
16143,2,7,2419.84
13243,202,1,600.07


In [7]:
from scipy.stats import zscore
import pandas as pd

# Convert to pandas
rfm_pd = rfm.to_pandas()

# Remove outliers
rfm_pd = rfm_pd[(zscore(rfm_pd[['Recency','Frequency','Monetary']]).abs() < 3).all(axis=1)]

rfm_pd.head()

AttributeError: 'numpy.ndarray' object has no attribute 'abs'

In [37]:
rfm_clean = pl.from_pandas(rfm_pd)

NameError: name 'rfm_pd' is not defined

In [ ]:
rfm_clean.write_csv("../data/processed_features.csv")